In [12]:
import os, shutil, stat
from pathlib import Path

def on_rm_error(func, path, excinfo):
    # try to change file permission and retry
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print("on_rm_error could not remove:", path, type(e), e)

template_path = Path(r"C:\Python\Personal\proj6\codes\runs\pest\baseline_template")
os.chdir(Path.home())

print("Attempting robust rmtree on:", template_path)
try:
    shutil.rmtree(template_path, onerror=on_rm_error)
    print("Removed", template_path)
except Exception as e:
    print("Still failed:", type(e), e)

Attempting robust rmtree on: C:\Python\Personal\proj6\codes\runs\pest\baseline_template
Removed C:\Python\Personal\proj6\codes\runs\pest\baseline_template


In [13]:
import os
import sys
import yaml
import shutil
import subprocess
import pyemu
import copy
from pathlib import Path

def prepare_MOU_files(objectives='model_misfit',
                      scenario_name=None,
                      param_set_selector=None,
                      use_log_transform=True,
                      r_lb=0.1,
                      r_ub=1e5):
    """
    Build template folder and PST for estimating well radial_dist (__r).
    - Measured fluxes are generated by running the forward model once with numeric inputs.
    - param_set_selector: None, int (index), or str (name) to pick param_set in base proj6.yml.
    - use_log_transform: if True, sets partrans='log' for radial parameters.
    - r_lb, r_ub: parameter bounds for radii (same units as radial_dist in proj6.yml).
    Returns: (scenario_name, template_path_str)
    """
    # 1. Setup Absolute Paths (adjust root if needed)
    root = Path(r"C:\Python\Personal\proj6\codes").resolve()
    pycap_run_name = "proj6"
    python_exe = sys.executable  # Path to python.exe to use when running the forward model

    if scenario_name is None:
        scenario_name = objectives

    parent_run_path = root / "runs"
    base_run_path = parent_run_path / "base"
    pest_path = parent_run_path / "pest"
    template_path = pest_path / f"{scenario_name}_template"
    script_source = root / "scripts" / "run_theis_forward_heads.py"

    # 2. Clean & Recreate Template Folder
    if template_path.exists():
        import time
        try:
            shutil.rmtree(template_path)
        except PermissionError:
            time.sleep(1)
            shutil.rmtree(template_path)
    template_path.mkdir(parents=True, exist_ok=True)

    # 3. Read base proj6.yml and (optionally) apply param_set
    base_yml_path = base_run_path / f"{pycap_run_name}.yml"
    if not base_yml_path.exists():
        raise FileNotFoundError(f"Base proj6.yml not found at {base_yml_path}")

    with open(base_yml_path, 'r') as ifp:
        indat = yaml.safe_load(ifp)

    # apply param_set_selector if requested
    if param_set_selector is not None:
        ps = indat.get('param_sets', [])
        if isinstance(param_set_selector, int):
            if param_set_selector < 0 or param_set_selector >= len(ps):
                raise IndexError("param_set_selector index out of range")
            sel = ps[param_set_selector]
        else:
            matches = [p for p in ps if p.get('name') == str(param_set_selector)]
            if not matches:
                raise KeyError(f"No param_set named '{param_set_selector}'")
            sel = matches[0]
        for k in ('T', 'S', 't_eval'):
            if k in sel:
                indat[k] = sel[k]

    # 4. Create a numeric copy of proj6.yml (with numeric radial_dist) to run the forward model
    #    We'll run the forward model from template folder but with this numeric file first.
    numeric_cfg = copy.deepcopy(indat)
    # ensure wells have numeric radial_dist values (they currently do in base file)
    well_keys = [i for i in numeric_cfg.keys() if i.startswith('well_')]
    if not well_keys:
        raise RuntimeError("No well_ entries found in base proj6.yml")

    # 5. Write numeric proj6.yml into the template folder for the test forward-run
    numeric_proj6_path = template_path / f"{pycap_run_name}_numeric.yml"
    with open(numeric_proj6_path, 'w') as ofp:
        yaml.dump(numeric_cfg, ofp, default_flow_style=False, sort_keys=False)

    # 6. Copy run_theis_forward.py into template folder (so the exact script will be used)
    if not Path(script_source).exists():
        raise FileNotFoundError(f"Model script not found at {script_source}")
    shutil.copy(script_source, template_path / "run_theis_forward_heads.py")

    # 7. Run forward model once to produce measured fluxes (allobs_meas.out)
    cwd = os.getcwd()
    os.chdir(template_path)
    try:
        # run python_proj6_numeric -> it expects proj6.yml name; our run_theis_forward supports
        # passing a selector, but easiest is to call the script with explicit config filename.
        # We'll run the script by invoking python and passing the numeric yaml as the config.
        # To support that the script needs to accept a config filename; if not, we temporarily
        # create proj6.yml with numeric values and run script normally.

        # Create a temporary proj6.yml (numeric) the model will read
        temp_infile = template_path / f"{pycap_run_name}.yml"
        with open(temp_infile, 'w') as ofp:
            yaml.dump(numeric_cfg, ofp, default_flow_style=False, sort_keys=False)

        # Run the model (use the python_exe that will also be used in MODEL_COMMAND)
        proc = subprocess.run([python_exe, "run_theis_forward_heads.py"],
                              stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if proc.returncode != 0:
            # show stderr for debugging
            raise RuntimeError(f"Forward model run failed:\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}")

        # If model ran, read the produced allobs.out (these are our measured fluxes)
        meas_path = template_path / "allobs.out"
        if not meas_path.exists():
            # If model didn't produce it, error
            raise RuntimeError("Forward model did not produce allobs.out when run numerically.")
        # Read measured values and keep them for later
        measured_lines = meas_path.read_text().splitlines()
        measured = {}
        for ln in measured_lines:
            parts = ln.split()
            if len(parts) >= 2:
                measured[parts[0]] = float(parts[1])
    finally:
        # remove the numeric proj6.yml we created (we will write a tokenized version later)
        # but keep run_theis_forward.py copy and allobs.out (we want measured)
        if (template_path / f"{pycap_run_name}.yml").exists():
            # leave measured allobs.out in place
            pass
        os.chdir(cwd)

    # 8. Now parameterize radial_dist with tokens in indat, write .tpl and tokenized in-file
    #    Replace radial_dist with token like ~well_1__r~ in indat.
    for k in well_keys:
        # build token name, centered in field (same style as before)
        indat[k]['radial_dist'] = f"~{k + '__r':^20}~"

    # Write the tpl (template) file
    with open(template_path / f"{pycap_run_name}.yml.tpl", 'w') as ofp:
        ofp.write('ptf ~\n')
        yaml.dump(indat, ofp, default_flow_style=False, sort_keys=False)

    # Write the tokenized proj6.yml (this is the in-file PEST will substitute into)
    with open(template_path / f"{pycap_run_name}.yml", 'w') as ofp:
        yaml.dump(indat, ofp, default_flow_style=False, sort_keys=False)

    # 9. Create instruction file (maps model output to observations) and measured allobs.out (overwrite)
    with open(template_path / 'allobs.out.ins', 'w') as ofp:
        ofp.write('pif ~\n')
        for k in sorted(well_keys):
            ofp.write(f'l1 w !{k}!\n')

    # overwrite allobs.out with measured values we just generated
    with open(template_path / 'allobs.out', 'w') as ofp:
        for k in sorted(measured.keys()):
            ofp.write(f"{k} {measured[k]}\n")

    # 10. Build the PST Object from the files in the template folder
    cwd = os.getcwd()
    os.chdir(template_path)
    try:
        pst = pyemu.Pst.from_io_files(
            tpl_files=[f"{pycap_run_name}.yml.tpl"],
            in_files=[f"{pycap_run_name}.yml"],
            ins_files=["allobs.out.ins"],
            out_files=["allobs.out"]
        )

        # 11. Set model command so PEST runs the copied script with the same python
        pst.model_command = [f"{python_exe} run_theis_forward_heads.py"]

        # 12. Configure parameters: set __r parameters adjustable, use log transform if requested
        # Identify radial parameters ending with __r
        r_mask = pst.parameter_data.parnme.str.endswith("__r")
        if use_log_transform:
            pst.parameter_data.loc[r_mask, "partrans"] = "log"
        else:
            pst.parameter_data.loc[r_mask, "partrans"] = "none"
        # set bounds (parlbnd/parubnd) and initial parval1 remain the values read from the tokenized in-file
        pst.parameter_data.loc[r_mask, "parlbnd"] = r_lb
        pst.parameter_data.loc[r_mask, "parubnd"] = r_ub
        # ensure they are adjustable (set parval1 from numeric_cfg if needed)
        # pyemu reads parval1 from the in-file numeric values before tokenization;
        # to be safe, set parval1 explicitly from the numeric_cfg we used earlier:
        for k in well_keys:
            parn = f"{k}__r"
            if parn in list(pst.parameter_data.parnme):
                # original numeric value was numeric_cfg[k]['radial_dist']
                val = float(numeric_cfg[k]['radial_dist'])
                pst.parameter_data.loc[pst.parameter_data.parnme == parn, "parval1"] = val

        # 13. Observation settings: keep observation group and weights
        pst.observation_data.loc[:, "obgnme"] = "less_misfit"
        # weight: set to 1.0 or 1/sigma if you have measurement errors
        pst.observation_data.loc[:, "weight"] = 1.0

        # 14. Control settings: turn on estimation (positive noptmax)
        pst.control_data.noptmax = 2  # change to how many iterations you want

        # 15. Write the PST
        pst.write(f"{scenario_name}.pst")
    finally:
        os.chdir(cwd)

    # 16. Return scenario name and template path
    return scenario_name, str(template_path)

In [14]:
import subprocess, os
name, path = prepare_MOU_files(scenario_name='baseline', param_set_selector=0)
os.chdir(path)
pest_exe = r"C:\Python\Personal\proj6\codes\binaries\PESTPP\windows\pestpp-mou.exe"
proc = subprocess.run([pest_exe, f"{name}.pst"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print(proc.returncode)
print(proc.stdout)
print(proc.stderr)

noptmax:2, npar_adj:2, nnz_obs:2
0


             pestpp-mou: multi-objective optimization under uncertainty

                   by the PEST++ development team

...processing command line: ' C:\Python\Personal\proj6\codes\binaries\PESTPP\windows\pestpp-mou.exe baseline.pst'
...using serial run manager


version: 5.2.24
binary compiled on Nov 12 2025 at 12:05:03
using control file: "baseline.pst"
in directory: "C:\Python\Personal\proj6\codes\runs\pest\baseline_template"
on host: "TTL-HBJ6674"
on a(n) windows operating system
with release configuration
started at 03/13/26 15:20:08

processing control file baseline.pst
Note: 3 unused lines in pest control file, see rec file...
checking model IO files...done
              starting serial run manager ...


  ---  initializing MOEA process  ---  
...using 'nsga2' env selector
...using binary tournament mating pool selector
...'mou_save_population_every' less than/equal to zero, not saving generation-specific populations (and archives)
using 

In [15]:
import pyemu
pst = pyemu.Pst("baseline.pst")   # or f"{name}.pst"
r_params = pst.parameter_data.loc[pst.parameter_data.parnme.str.endswith("__r"), ["parnme","parval1","parlbnd","parubnd"]]
print(r_params)

              parnme  parval1  parlbnd   parubnd
parnme                                          
well_1__r  well_1__r  2236.07      0.1  100000.0
well_2__r  well_2__r  6324.56      0.1  100000.0


In [16]:
# forward_check.py
import yaml, subprocess, sys
from pathlib import Path
import numpy as np

# load pst to get estimated params
import pyemu
pst = pyemu.Pst("baseline.pst")   # or your pst filename

# get estimated radii parameters (parval1)
r_params = pst.parameter_data.loc[pst.parameter_data.parnme.str.endswith("__r"), ["parnme","parval1"]]
print("Estimated parameters:")
print(r_params)

# write proj6.yml with these estimated radii (replace tokens with values)
cfg = yaml.safe_load(Path("proj6.yml").read_text())  # tokenized file
for parn,val in zip(r_params.parnme, r_params.parval1):
    # parn like "well_1__r" -> well key "well_1"
    well = parn.replace("__r","")
    if well in cfg:
        cfg[well]['radial_dist'] = float(val)

# save a temporary proj6_est.yml and run model
tmp = Path("proj6_est.yml")
with tmp.open('w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

# run model using same python as in PST
python_exe = pst.model_command[0].split()[0]  # crude parse: "C:\...\python.exe run_theis_forward.py"
proc = subprocess.run([python_exe, "run_theis_forward_heads.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print("Model returncode:", proc.returncode)
if proc.stdout: print("MODEL STDOUT:\n", proc.stdout)
if proc.stderr: print("MODEL STDERR:\n", proc.stderr)

# read model predictions and observations
pred = {}
for ln in Path("allobs.out").read_text().splitlines():
    k,v = ln.split()[:2]
    pred[k] = float(v)

obs = {}
# measured allobs.out was used as obs file; load the same file we kept
for ln in Path("allobs.out").read_text().splitlines():
    k,v = ln.split()[:2]
    obs[k] = float(v)

# compute residuals
print("\nResiduals (obs - pred):")
for k in sorted(obs):
    r = obs[k] - pred.get(k,0.0)
    print(k, "obs=",obs[k], "pred=", pred.get(k, None), "resid=", r)

Estimated parameters:
              parnme  parval1
parnme                       
well_1__r  well_1__r  2236.07
well_2__r  well_2__r  6324.56
Model returncode: 0

Residuals (obs - pred):
well_1 obs= 27399.848865849457 pred= 27399.848865849457 resid= 0.0
well_2 obs= 0.0 pred= 0.0 resid= 0.0
